## Celula 1 -- Instalacao

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.55.2
!pip install --no-deps trl==0.22.2
!pip install scikit-learn
print("Done")

## Celula 2 -- Carregar modelo e QLoRA

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

MAX_SEQ_LENGTH = 512
SEED = 42

# Carrega o modelo base -- get_peft_model adiciona os adaptadores LoRA treinaveis
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,
    dtype          = None,
)

#model = FastLanguageModel.get_peft_model(
    model,
    r                        = 32,
    target_modules           = ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj"],
    lora_alpha               = 32,
    lora_dropout             = 0,
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",
    random_state             = SEED,
    use_rslora               = False,
    loftq_config             = None,
)

# Chat template -- chamado UMA unica vez aqui
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

# Verificacao de parametros treinaveis
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Parametros treinaveis: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Celula 12 -- Prompts

In [ ]:
def ask_qwen(user_prompt: str, system_prompt: str = None, max_new_tokens: int = 512) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,        # agora ajustável
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.eos_token_id],
        )
    
    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## Celula 14 -- Execução de Prompts

In [ ]:
# Texto do problema (copiado da sua descrição)
problem_text = """time limit per test: 1 second
memory limit per test: 256 megabytes
You are given two integers n and k.
Construct a binary string s of length n, such that both of the following conditions hold:
The absolute difference between the number of characters 0 and the number of characters 1 in s is at most 1.
There are exactly k pairs of adjacent equal characters in s. Formally, there are exactly k indices i(1≤i≤n−1) satisfying si=si+1.
Or determine that no such string exists.
A binary string is a string where each character is either 0 or 1.

Input: Each test contains multiple test cases. The first line contains the number of test cases t (1≤t≤1000). The description of the test cases follows. The only line of each test case contains two integers n and k (2≤n≤2⋅10^5, 0≤k≤n−1). It is guaranteed that the sum of n over all test cases does not exceed 2⋅10^5.

Output: For each test case, output a binary string s of length n — the string you constructed. Print −1 if such a string does not exist. If there are multiple answers, you may output any of them.

Example
Input
8
5 2
4 3
6 1
5 0
7 3
4 2
3 2
7 4
Output
01110
-1
101001
01010
0100011
0011
-1
0111000"""

# Definição dos prompts
prompt_code = f"""Write a Python solution for the following competitive programming problem. 
Return only the code, no explanation.

{problem_text}"""

prompt_pseudocode = f"""Write a pseudocode solution for the following competitive programming problem. 
Return only the pseudocode, no explanation.

{problem_text}"""

prompt_semantic = f"""Rewrite the following competitive programming problem in plain English, 
highlighting the key algorithmic concepts and what the task asks for. 
Do not solve it, just explain the problem clearly.

{problem_text}"""

prompt_topics = f"""List the algorithmic topics that a student needs to know to solve this problem. 
Return only a comma-separated list of topics, no explanation.

{problem_text}"""

# Dicionário de prompts
prompts = {
    "Código Python": prompt_code,
    "Pseudocódigo": prompt_pseudocode,
    "Transformação semântica": prompt_semantic,
    "Tópicos": prompt_topics,
}

# Limites de tokens por tipo de resposta
max_tokens = {
    "Código Python": 1024,
    "Pseudocódigo": 1024,
    "Transformação semântica": 200,
    "Tópicos": 200,
}

# Execução
results = {}
for label, prompt in prompts.items():
    print(f"=== {label} ===")
    answer = ask_qwen(prompt, max_new_tokens=max_tokens[label])
    results[label] = answer
    print(answer)
    print("\n" + "-"*60 + "\n")

In [ ]:
import os
from datetime import datetime

# Pasta base para salvar os resultados
base_dir = "/kaggle/working/baseline_results"
os.makedirs(base_dir, exist_ok=True)

# Identificador do modelo (ajuste conforme o modelo carregado)
model_name = "qwen3-4b-base"   # mude para "qwen3-4b-v1" ou "qwen3-4b-v2" conforme necessário
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Nome da subpasta para este experimento
experiment_dir = os.path.join(base_dir, f"{model_name}_{timestamp}")
os.makedirs(experiment_dir, exist_ok=True)

# Para cada resposta, cria um arquivo .txt
for label, answer in results.items():
    # Nome do arquivo com espaços substituídos por underscore
    filename = label.replace(" ", "_").lower() + ".txt"
    filepath = os.path.join(experiment_dir, filename)

    # Conteúdo do arquivo: inclui o prompt e a resposta
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"Modelo: {model_name}\n")
        f.write(f"Data: {timestamp}\n")
        f.write(f"Tarefa: {label}\n")
        f.write("="*60 + "\n")
        f.write("PROMPT UTILIZADO:\n")
        f.write(prompts[label] + "\n\n")
        f.write("="*60 + "\n")
        f.write("RESPOSTA DO MODELO:\n")
        f.write(answer + "\n")

print(f"Resultados salvos em: {experiment_dir}")